<a href="https://colab.research.google.com/github/emilsar/NLP-TCGA/blob/main/notebooks/cancer_type/1-Compile_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Compiling the Dataset

Join the pathology report text to its cancer-type label, using the patient barcode as the key.

*NLP on TCGA pathology reports. Run the setup cell below first, then work down the notebook.*


In [ ]:
# === SETUP — run this cell first ===
# On Google Colab this downloads the course repo (notebooks + data) so that the
# relative paths further down ("../../data") work. Running locally it does nothing.
import os, sys, subprocess

REPO_URL = "https://github.com/emilsar/NLP-TCGA.git"
REPO_DIR = "/content/NLP-TCGA"
NB_SUBDIR = "notebooks/cancer_type"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        print("Downloading course data (~100 MB, takes about a minute)...")
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )
    os.chdir(os.path.join(REPO_DIR, NB_SUBDIR))

print("Working directory:", os.getcwd())
print("Data folder found:", os.path.isdir("../../data"))


In [1]:
# pandas is the library for working with tables of data (called "DataFrames").
# os builds file paths that work the same on Windows, Mac and Colab.
import os
import pandas as pd

# Data loading

In [2]:
# Where the data lives, written relative to this notebook.
# "../.." means go up two folders: from notebooks/cancer_type/ to the repo root.
# Defining it once means only one line changes if the data ever moves.
data_dir = "../../data"

## Corpus

In [3]:
# Read the pathology reports into a DataFrame.
# sep=',' says columns are separated by commas.
# header=0 says the first row holds the column names.
# The file is ~34 MB, so give it a few seconds.
df_corpus = pd.read_csv(
    os.path.join(data_dir, "corpus", "TCGA_Reports.csv"),
    sep=',',
    header=0
)

In [4]:
# .shape reports (number of rows, number of columns). One row = one patient's report.
df_corpus.shape

(9523, 2)

In [5]:
# .head() shows the first 5 rows. Always look at data before doing anything to it.
df_corpus.head()

,patient_filename,text
0,TCGA-BP-5195.25c0b433-5557-4165-922e-2c1eac9c26f0,Date of Recelpt: Clinical Diagnosis & History:...
1,TCGA-D7-8573.b7306a47-697d-4ed3-bbe1-81d49674a8f8,"Material: 1) Material: stomach, Method of coll..."
2,TCGA-EI-7004.13591eed-30e5-47a3-91be-7a370663d2d4,page 1 / 1. copy No. 3. Examination: Histopath...
3,TCGA-EB-A82B.23E186C6-739C-4EF1-8788-79AA89C6E87A,Patient ID: Gross Description: A mass is locat...
4,TCGA-A6-3808.e1505f65-72ef-438d-a5e1-93ed8bf6635d,SPECIMEN. Right colon. CLINICAL NOTES. PRE-OP ...


Extract TCGA patient IDs:

In [6]:
# The patient barcode is hiding at the front of the filename, before the '.'.
# .apply() runs the small function on every row; .split('.') cuts the text at the dot,
# and [0] keeps the piece before it:
#   "TCGA-BP-5195.25c0b433-..."  ->  "TCGA-BP-5195"
# That barcode is what lets us match labels to reports further down.
df_corpus["patient_id"] = df_corpus["patient_filename"].apply(lambda x: x.split('.')[0])

In [7]:
# assert means "stop everything if this is not true" - a safety net for our assumptions.
# Here: check no filename appears twice.
assert not df_corpus["patient_filename"].duplicated().any()

In [8]:
# And no patient appears twice, so one barcode always means exactly one report.
# That is what makes the barcode safe to use as a matching key.
assert not df_corpus["patient_id"].duplicated().any()

In [9]:
# Keep only the two columns we need. The double brackets [[...]] select a list of columns.
df_corpus = df_corpus[["patient_id", "text"]]

In [10]:
# Set the barcode as the DataFrame's index - its row labels.
# With barcodes as labels we can look rows up by patient instead of by position.
df_corpus.index = df_corpus["patient_id"].values

In [11]:
# Notice the labels down the left are now barcodes instead of 0, 1, 2, ...
df_corpus.head()

,patient_id,text
TCGA-BP-5195,TCGA-BP-5195,Date of Recelpt: Clinical Diagnosis & History:...
TCGA-D7-8573,TCGA-D7-8573,"Material: 1) Material: stomach, Method of coll..."
TCGA-EI-7004,TCGA-EI-7004,page 1 / 1. copy No. 3. Examination: Histopath...
TCGA-EB-A82B,TCGA-EB-A82B,Patient ID: Gross Description: A mass is locat...
TCGA-A6-3808,TCGA-A6-3808,SPECIMEN. Right colon. CLINICAL NOTES. PRE-OP ...


## Cancer type

In [12]:
# Now the labels: which cancer type each patient has.
# A separate file from a separate source - TCGA's clinical records, not the reports.
df_cancer_type = pd.read_csv(
    os.path.join(data_dir, "cancer_type", "tcga_patient_to_cancer_type.csv"),
    sep=',',
    header=0
)

In [13]:
# 11,160 patients here, more than we have reports for. The extras get dropped by the join.
df_cancer_type.shape

(11160, 2)

In [14]:
# Two columns: the barcode, and a short TCGA code such as ACC or BRCA.
df_cancer_type.head()

,patient_id,cancer_type
0,TCGA-OR-A5J1,ACC
1,TCGA-OR-A5J2,ACC
2,TCGA-OR-A5J3,ACC
3,TCGA-OR-A5J4,ACC
4,TCGA-OR-A5J5,ACC


In [15]:
# Same safety check as before: one row per patient.
assert not df_cancer_type["patient_id"].duplicated().any()

In [16]:
# Index this table by barcode too, so the two tables can be lined up.
df_cancer_type.index = df_cancer_type["patient_id"].values

In [17]:
# Both tables are now labelled by patient barcode - ready to join.
df_cancer_type.head()

,patient_id,cancer_type
TCGA-OR-A5J1,TCGA-OR-A5J1,ACC
TCGA-OR-A5J2,TCGA-OR-A5J2,ACC
TCGA-OR-A5J3,TCGA-OR-A5J3,ACC
TCGA-OR-A5J4,TCGA-OR-A5J4,ACC
TCGA-OR-A5J5,TCGA-OR-A5J5,ACC


# Data merging

We combine both datasets into a single dataframe:

In [18]:
# Before joining, check that every report actually has a label.
# .isin(...) asks "is each value found in that other column?"
# .all() asks "was that true every single time?" 
assert df_corpus["patient_id"].isin(df_cancer_type["patient_id"]).all()

In [19]:
# This is the join. For each barcode in df_corpus, pull that patient's cancer type.
# .loc[labels, "column"] looks rows up by index label - which is why we set the index above.
df_corpus["cancer_type"] = df_cancer_type.loc[df_corpus.index, "cancer_type"]

In [20]:
# Text and label now sit side by side. This pairing is what supervised learning needs.
df_corpus.head()

,patient_id,text,cancer_type
TCGA-BP-5195,TCGA-BP-5195,Date of Recelpt: Clinical Diagnosis & History:...,KIRC
TCGA-D7-8573,TCGA-D7-8573,"Material: 1) Material: stomach, Method of coll...",STAD
TCGA-EI-7004,TCGA-EI-7004,page 1 / 1. copy No. 3. Examination: Histopath...,READ
TCGA-EB-A82B,TCGA-EB-A82B,Patient ID: Gross Description: A mass is locat...,SKCM
TCGA-A6-3808,TCGA-A6-3808,SPECIMEN. Right colon. CLINICAL NOTES. PRE-OP ...,COAD


We add a column with the full name of the cancer type. This information was downloaded from here: https://github.com/KatherLab/cancer-metadata/blob/main/tcga/tcga-tumor-types.csv

In [21]:
# A small lookup table: TCGA code -> full cancer name.
# Note sep=';' - this file uses semicolons, not commas. Always check the separator.
df_cancer_name = pd.read_csv(
    os.path.join(data_dir, "cancer_type", "tcga-tumor-types.csv"),
    sep=';',
    header=0
)

In [22]:
# 37 codes are listed here; only 32 of them actually appear in our reports.
df_cancer_name.shape

(37, 2)

In [23]:
# For example BRCA = "Breast invasive carcinoma".
df_cancer_name.head()

,Study Abbreviation,Study Name
0,LAML,Acute Myeloid Leukemia
1,ACC,Adrenocortical carcinoma
2,BLCA,Bladder Urothelial Carcinoma
3,LGG,Brain Lower Grade Glioma
4,BRCA,Breast invasive carcinoma


In [24]:
# Turn the two-column table into a Python dictionary:
#   {'BRCA': 'Breast invasive carcinoma', 'ACC': 'Adrenocortical carcinoma', ...}
# Dictionaries look a value up by its key, which is exactly what the next cell needs.
# Convert DataFrame into dictionary
assert not df_cancer_name["Study Abbreviation"].duplicated().any()
df_cancer_name.index = df_cancer_name["Study Abbreviation"].values
dict_cancer_name = df_cancer_name["Study Name"].to_dict()

In [25]:
# Check every code in our data has an entry in the lookup, before relying on it.
assert df_corpus["cancer_type"].isin(list(dict_cancer_name.keys())).all()

In [26]:
# Add a human-readable column by looking up each code in the dictionary.
# This column is only for labelling plots later - it is NOT used for modelling.
df_corpus["cancer_type_name"] = df_corpus["cancer_type"].apply(lambda x: dict_cancer_name[x])

In [27]:
# Save the assembled dataset - this file is what notebook 2 opens.
# index=False means don't write the row labels out as an extra column.
df_corpus.to_csv(
    os.path.join(data_dir, "cancer_type", "tcga_reports_cancer_type.csv"),
    sep=',',
    index=False,
    header=True
)